In [1]:
import numpy as np
import pandas as pd
import pywt
from scipy.stats import entropy
from scipy.signal import find_peaks
from sklearn.preprocessing import StandardScaler

# 提取三个特征模块

输出：

features_module1_dwt：DWT 多尺度特征

features_module2_phase：24h 相位特征

features_module3_amplitude：幅值/消耗特征

features_fused：加权融合后的最终特征空间

## 0.读取数据

In [3]:
# ----------------------------------------------------
# 1. 读取采样后的 CSV（每一列是一个负荷节点）
# ----------------------------------------------------
CSV_PATH = r"..\K-shape\kshape_results_8784_0Start\sampled_8784_0Start.csv"
PERIOD = 24

df = pd.read_csv(CSV_PATH, header=0)

# ----------------------------------------------------
# 2. 周期对齐缺失值填补（24h）
# ----------------------------------------------------
missing = df.isna().sum().sum()

if missing > 0:
    print(f"发现 {missing} 个缺失值，使用周期对齐插值 (period={PERIOD}) 填补。")
    
    # 对每一列进行处理
    for col_idx, col_name in enumerate(df.columns):
        series = df[col_name].values
        nan_idx = np.where(np.isnan(series))[0]
        
        if len(nan_idx) == 0:
            continue
            
        print(f"列 `{col_name}` (第 {col_idx + 1} 列) 有 {len(nan_idx)} 个缺失值")
        
        # 第一次迭代：填补所有可能的缺失值
        for t in nan_idx:
            # 确保前后24小时的数据都存在且在边界内
            if t - PERIOD >= 0 and t + PERIOD < len(series):
                if not (np.isnan(series[t - PERIOD]) or np.isnan(series[t + PERIOD])):
                    series[t] = 0.5 * (series[t - PERIOD] + series[t + PERIOD])
        
        # 多次迭代，直到所有可能的缺失值都被填满
        iteration = 1
        while np.isnan(series).any() and iteration < 10:  # 最多迭代10次防止死循环
            remaining_before = np.isnan(series).sum()
            current_nan = np.where(np.isnan(series))[0]
            
            for t in current_nan:
                if t - PERIOD >= 0 and t + PERIOD < len(series):
                    if not (np.isnan(series[t - PERIOD]) or np.isnan(series[t + PERIOD])):
                        series[t] = 0.5 * (series[t - PERIOD] + series[t + PERIOD])
            
            remaining_after = np.isnan(series).sum()
            filled = remaining_before - remaining_after
            iteration += 1
            
            if filled == 0:  # 如果没有新的填补，跳出循环避免无限循环
                break
        
        # 如果还有缺失值，使用线性插值作为备选方案
        if np.isnan(series).any():
            remaining = np.isnan(series).sum()
            print(f"  列 `{col_name}` 仍有 {remaining} 个缺失值，使用线性插值填补")
            # 创建临时Series进行线性插值
            temp_series = pd.Series(series)
            series = temp_series.interpolate(method='linear', limit_direction='both').values
            
        df[col_name] = series
    
    print("缺失值填补完成")
else:
    print("未发现缺失值，无需填补。")

# ----------------------------------------------------
# 3. 构造与你原来完全一致的变量
# ----------------------------------------------------
data = df.values.T        # shape: (n_nodes, 1440)
node_names = df.columns.tolist()
n_nodes = data.shape[0]

print("数据读取与缺失值填补完成：")
print("data shape:", data.shape)
print("n_nodes:", n_nodes)

发现 67 个缺失值，使用周期对齐插值 (period=24) 填补。
列 `Robin_office_Soledad` (第 42 列) 有 2 个缺失值
列 `Mouse_health_Buddy` (第 62 列) 有 50 个缺失值
列 `Mouse_health_Modesto` (第 63 列) 有 4 个缺失值
列 `Mouse_health_Ileana` (第 66 列) 有 11 个缺失值
缺失值填补完成
数据读取与缺失值填补完成：
data shape: (68, 8784)
n_nodes: 68


## 模块 1：DWT 多尺度特征（周期 & 波动形态）

提取内容（对DWT分解后的每一层提取）：

mean

std

energy

entropy

输出维度：维度 = (7 层 + 1 个 A_L) × 4 = 32 维

In [4]:
def extract_dwt_features(
    ts,
    wavelet="sym5",
    level=7
):
    """
    对单条时间序列提取 DWT 多尺度统计特征
    """
    coeffs = pywt.wavedec(ts, wavelet=wavelet, level=level)

    features = []

    # coeffs: [A_L, D_L, D_{L-1}, ..., D_1]
    for c in coeffs:
        c = np.asarray(c)

        mean_c = np.mean(c)
        std_c = np.std(c)
        energy_c = np.sum(c ** 2)

        # 归一化后算熵（防止数值问题）
        prob = np.abs(c)
        prob = prob / (np.sum(prob) + 1e-12)
        entropy_c = entropy(prob)

        features.extend([mean_c, std_c, energy_c, entropy_c])

    return np.array(features)


In [5]:
features_module1_dwt = np.vstack([
    extract_dwt_features(data[i])
    for i in range(n_nodes)
])

print("Module 1 (DWT) feature shape:", features_module1_dwt.shape)


Module 1 (DWT) feature shape: (68, 32)


## 模块 2：24h 日内相位特征

核心思想：

1440 h → reshape 为 (60 天 × 24 h)

得到“典型日负荷曲线”

在 24h 曲线上提取峰谷与相位信息

输出维度：6
（peak_hour,
valley_hour,
peak_value,
valley_value,
num_peaks,
peak_to_valley_ratio）

In [6]:
def extract_phase_features(ts):
    """
    提取日内相位与峰谷特征
    """
    ts = ts.reshape(-1, 24)   # (60, 24)
    daily_profile = ts.mean(axis=0)  # 典型 24h 曲线

    # 主峰检测
    peaks, _ = find_peaks(daily_profile)
    valleys, _ = find_peaks(-daily_profile)

    # 主峰
    if len(peaks) > 0:
        main_peak_idx = peaks[np.argmax(daily_profile[peaks])]
        peak_hour = main_peak_idx
        peak_value = daily_profile[main_peak_idx]
        num_peaks = len(peaks)
    else:
        peak_hour = -1
        peak_value = daily_profile.max()
        num_peaks = 0

    # 主谷
    if len(valleys) > 0:
        main_valley_idx = valleys[np.argmin(daily_profile[valleys])]
        valley_hour = main_valley_idx
        valley_value = daily_profile[main_valley_idx]
    else:
        valley_hour = -1
        valley_value = daily_profile.min()

    # 峰谷比
    peak_to_valley_ratio = (
        peak_value / (valley_value + 1e-6)
        if valley_value != 0 else 0
    )

    return np.array([
        peak_hour,
        valley_hour,
        peak_value,
        valley_value,
        num_peaks,
        peak_to_valley_ratio
    ])


In [7]:
features_module2_phase = np.vstack([
    extract_phase_features(data[i])
    for i in range(n_nodes)
])

print("Module 2 (Phase) feature shape:", features_module2_phase.shape)


Module 2 (Phase) feature shape: (68, 6)


## 模块 3：幅值 & 消耗水平特征

提取内容：

mean / max / min

p95 / p5

range

coefficient of variation

输出维度：7

In [8]:
def extract_amplitude_features(ts):
    mean_load = np.mean(ts)
    std_load = np.std(ts)
    max_load = np.max(ts)
    min_load = np.min(ts)

    p95 = np.percentile(ts, 95)
    p5 = np.percentile(ts, 5)

    load_range = max_load - min_load
    cv = std_load / (mean_load + 1e-6)

    return np.array([
        mean_load,
        max_load,
        min_load,
        p95,
        p5,
        load_range,
        cv
    ])


In [9]:
features_module3_amplitude = np.vstack([
    extract_amplitude_features(data[i])
    for i in range(n_nodes)
])

print("Module 3 (Amplitude) feature shape:", features_module3_amplitude.shape)


Module 3 (Amplitude) feature shape: (68, 7)


## 模块 4：模块内标准化 + 加权融合

融合出用于聚类的特征空间

标准化：

模块能量归一：否则高维的dwt主导了特征空间

In [10]:
scaler_dwt = StandardScaler()
scaler_phase = StandardScaler()
scaler_amp = StandardScaler()

features_dwt_std = scaler_dwt.fit_transform(features_module1_dwt)
features_phase_std = scaler_phase.fit_transform(features_module2_phase)
features_amp_std = scaler_amp.fit_transform(features_module3_amplitude)

features_dwt_std /= np.sqrt(features_dwt_std.shape[1])   # √32
features_phase_std /= np.sqrt(features_phase_std.shape[1]) # √6
features_amp_std /= np.sqrt(features_amp_std.shape[1])   # √7


权重设置

In [11]:
w_dwt   = 0.6
w_phase = 1.8
w_amp   = 1.0


融合为45维特征空间

In [12]:
features_fused = np.hstack([
    w_dwt * features_dwt_std,
    w_phase * features_phase_std,
    w_amp * features_amp_std
])

print("Final fused feature shape:", features_fused.shape)


Final fused feature shape: (68, 45)


输出为DF，保存为csv

In [13]:
features_fused_df = pd.DataFrame(
    features_fused,
    index=node_names
)

features_fused_df.to_csv("Multi-Feature_result/fused_features_for_clustering_1Y_Weighted.csv")
